# Emerging Technologies

In [ ]:
import qiskit
from qiskit import QuantumCircuit
import qiskit_aer as aer
import numpy as np
import random
import itertools

## Problem 1: Generating Random Boolean Functions

In [10]:
def random_constant_balanced():
    """Returns a randomly chosen constant or balanced function."""
    
    if random.choice([True, False]):
        # Constant function
        constant_value = random.choice([True, False])
        return lambda a, b, c, d: constant_value
    else:
        # Balanced function - picks 8 of 16 inputs to return True
        true_inputs = set(random.sample(range(16), 8))
        
        def balanced(a, b, c, d):
            # Convert 4 bools to number 0-15
            n = (8*a + 4*b + 2*c + d)
            return n in true_inputs
        
        return balanced

# Test the function
f = random_constant_balanced()
print(f(True, True, False, True))  # Example: f(1,1,0,1)

False


## Problem 2: Classical Testing for Function Type

In [ ]:
def determine_constant_early_exit(f):
    """Check until we find a difference.
       if we find a difference, return 'balanced'.
       if we never find a difference, return 'constant'."""
    
    # Get the first result as our reference
    first_result = f(False, False, False, False)
    
    # Check all other inputs
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    # Skip the first one we already checked
                    if a == b == c == d == False:
                        continue
                    
                    result = f(a, b, c, d)
                    if result != first_result:
                        # Found a difference!
                        return "balanced"
    
    # Never found a difference
    return "constant"

In [ ]:
def determine_constant_all(f):
    """Count how many return True.
       If count is 0 or 16, return 'constant'.
       If count is 8, return 'balanced'."""
    
    true_count = 0
    
    # Test all 16 inputs
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    if f(a, b, c, d):
                        true_count += 1
    
    # Analyze the count
    if true_count == 0 or true_count == 16:
        return "constant"
    elif true_count == 8:
        return "balanced"
    else:
        # This should never happen with valid functions
        return "invalid function"

In [ ]:
def determine_constant_balanced(f):
    """
    Most efficient: maximum 9 calls needed.
    
    Key insight: If 9 inputs have the same result,
    it MUST be constant (can't be balanced).
    """
    
    # Get first result
    first_result = f(False, False, False, False)
    same_count = 1  # Count how many match first_result
    
    # Test remaining inputs
    for a in [False, True]:
        for b in [False, True]:
            for c in [False, True]:
                for d in [False, True]:
                    # Skip first input
                    if a == b == c == d == False:
                        continue
                    
                    result = f(a, b, c, d)
                    
                    if result != first_result:
                        # Found a difference - definitely balanced
                        return "balanced"
                    
                    same_count += 1
                    
                    # If 9 are the same, must be constant
                    if same_count >= 9:
                        return "constant"
    
    # All 16 were the same
    return "constant"

## Problem 3: Quantum Oracles

In [ ]:
def dj(num_qubits):
    """Generates a random constant or balanced function for the Deutsch-Jozsa algorithm.
    
    Args:
        num_qubits (int): Number of input qubits.
    """
    
    qc_dj = QuantumCircuit(num_qubits + 1)
    if np.random.randint(0, 2):
        # Flip output qubits with 50% chance
        qc_dj.x(num_qubits)
    if np.random.randint(0, 2):
        # return constant circuit with 50% chance.
        return qc_dj
 
    # If the "if" statement above was "TRUE" then we've returned the constant
    # function and the function is complete. If not, we proceed in creating our
    # balanced function. Everything below is to produce the balanced function:
 
    # select half of all possible states at random:
    on_states = np.random.choice(
        range(2**num_qubits),  # numbers to sample from
        2**num_qubits // 2,  # number of samples
        replace=False,  # makes sure states are only sampled once
    )
 
    def add_cx(qc_dj, bit_string):
        for qubit, bit in enumerate(reversed(bit_string)):
            if bit == "1":
                qc_dj.x(qubit)
        return qc_dj
 
    for state in on_states:
        # qc_dj.barrier()  # Barriers are added to help visualize how the functions are created. They can safely be removed.
        qc_dj = add_cx(qc_dj, f"{state:0b}")
        qc_dj.mcx(list(range(num_qubits)), num_qubits)
        qc_dj = add_cx(qc_dj, f"{state:0b}")
 
    # qc_dj.barrier()
 
    return qc_dj

n = 4  # number of input qubits

oracle = dj(n)

oracle.draw("mpl")


## Problem 4: Deutsch's Algorithm with Qiskit

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm